# 健身动作识别模型验证（Kaggle）

这个 notebook 做两件事：

1. **复现 MM-Fit 测试集指标**：加载仓库里的 `mmfit_bilstm_s42/best.pt`，在 Kaggle 私有数据集 `fitness-mmfit` 上跑 `test` split，应得到约 **0.9996 acc / 0.9997 macro-F1**。
2. **复现 Combined / Ensemble 指标**（可选）：需要把服务器上的 `data/processed/combined` 也上传为 Kaggle 数据集；应得到单模型约 **0.989 macro-F1**，top-3 ensemble 片段级约 **0.9931**。

> 注意：这是在**已切好的动作组**上验证，不是连续视频实时识别；真正的泛化要看新受试者/新摄像头视频。


In [ ]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path("/kaggle/working/repo")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/smileyan369/-.git", str(REPO)], check=True)
CODE = REPO / "谢金言" / "fitness-action"
os.chdir(CODE)
print("cwd:", os.getcwd())
print("torch:", subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"], capture_output=True, text=True).stdout)


## 1. 复现 MM-Fit 指标

先把 `kaggle_mmfit_processed.zip` 上传为 Kaggle 私有数据集（例如 dataset slug `fitness-mmfit`），
挂载后路径通常是 `/kaggle/input/fitness-mmfit/mmfit`。按实际路径修改 `DATA_MMFIT`。


In [ ]:
DATA_MMFIT = Path("/kaggle/input/fitness-mmfit/mmfit")
if not DATA_MMFIT.exists():
    import urllib.request, zipfile
    url = "https://github.com/smileyan369/-/releases/download/v1.0-data/kaggle_mmfit_processed.zip"
    zpath = Path("/kaggle/working/kaggle_mmfit_processed.zip")
    print("dataset not mounted; downloading fallback data...")
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall("/kaggle/working/mmfit_data")
    DATA_MMFIT = Path("/kaggle/working/mmfit_data/mmfit")
CKPT = CODE.parent / "cloud_results" / "runs" / "mmfit_bilstm_s42" / "best.pt"
OUT = Path("/kaggle/working/eval_mmfit")
assert DATA_MMFIT.exists(), f"missing {DATA_MMFIT}; upload the dataset or fix the path"
assert CKPT.exists(), f"missing {CKPT}"
subprocess.run([
    sys.executable, "-m", "src.evaluate",
    "--checkpoint", str(CKPT),
    "--data", str(DATA_MMFIT),
    "--split", "test",
    "--out", str(OUT),
], check=True)


In [ ]:
import pandas as pd
m = json.loads((OUT / "metrics.json").read_text())
print("accuracy:", m["accuracy"])
print("macro_f1:", m["macro_f1"])
print("per-class:")
display(pd.DataFrame(m["per_class"]).T)
display(pd.read_csv(OUT / "confusion_matrix.csv", index_col=0))


## 2. 复现 Combined / Ensemble（可选）

需要把服务器上的 `data/processed/combined` 上传为 Kaggle 数据集，例如
`/kaggle/input/fitness-combined/combined`。服务器路径：
`/root/autodl-tmp/fitness-action/data/processed/combined`（明早开机后可用）。


In [ ]:
DATA_COMBINED = Path("/kaggle/input/fitness-combined/combined")
if not DATA_COMBINED.exists():
    print("跳过：没有 combined 数据集；上传后把 DATA_COMBINED 指到实际路径")
else:
    OUT2 = Path("/kaggle/working/eval_combined")
    CKPT2 = CODE.parent / "cloud_results" / "runs" / "combined_tcn_s42" / "best.pt"
    subprocess.run([
        sys.executable, "-m", "src.evaluate",
        "--checkpoint", str(CKPT2),
        "--data", str(DATA_COMBINED),
        "--split", "test",
        "--out", str(OUT2),
    ], check=True)
    m2 = json.loads((OUT2 / "metrics.json").read_text())
    print("combined TCN accuracy:", m2["accuracy"], "macro_f1:", m2["macro_f1"])


In [ ]:
if DATA_COMBINED.exists():
    OUT3 = Path("/kaggle/working/eval_ensemble")
    subprocess.run([
        sys.executable, "scripts/ensemble_eval.py",
        "--data", str(DATA_COMBINED),
        "--runs", str(CODE.parent / "cloud_results" / "runs"),
        "--top-k", "3", "--stride", "4",
        "--out", str(OUT3),
    ], check=True)
    e = json.loads((OUT3 / "metrics.json").read_text())
    print("ensemble clip_acc:", e["clip_accuracy"], "clip_macro_f1:", e["clip_macro_f1"])
else:
    print("跳过 ensemble：没有 combined 数据集")


## 3. 怎么判断“验证通过”

- MM-Fit：accuracy 在 0.995–1.000、macro-F1 在 0.995–1.000，说明 checkpoint 和数据处理一致。
- Combined：单模型 macro-F1 约 0.98–0.99；ensemble 片段级 macro-F1 约 0.993。
- 如果 Kaggle 结果和报告差很多，优先检查：`classes.json` 的类别顺序、`manifest.csv` 的 split、`seq_path` 是否指向正确文件。
- 想验证“真实效果”，不要只用这个 test split：用手机录 3–5 段新视频（不同人/角度/灯光），走 `src.pose.extract` + 模型推理，才是真正的泛化测试。
